#### ***Positional Embedding***

##### **Position Embedding Tells the LLM where the each token in a sequence**

In [3]:
## lets create a embedding layer using vocab size 50,257 and dimensions of 256
import torch.nn as nn
vocab_size = 50257
out_dim = 256

embedding_layer = nn.Embedding(vocab_size,out_dim)

print(embedding_layer)


Embedding(50257, 256)


In [4]:
## tiktoken is a python library is used to convert the text into tokens,
#  and also convert those tokens into again text.
import tiktoken

In [5]:
## let's read the data from a file
with open("../Data/the-verdict.txt","r",encoding = "utf-8") as f:
    raw_text = f.read()

In [6]:
## We will use Pytorch built-in for dataset and dataloader classes to implement a Data Loader.

#How we are implementing
#1.Tokenize the Entire Text.
#2.Using sliding window means based on context size split's the data into input and output variable,
#output = input+[1] is used to predict the next token by one
#Return the Entire Dataset.
#Return the Single Row from the Dataset.

In [7]:
from torch.utils.data import DataLoader,Dataset
import torch


class GPTDatasetV1(Dataset):
    def __init__(self,text,tokenizer,max_length,stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(text,allowed_special={"<|endoftext|>"})

        #splits the token ids into input and output token ids based on context size
        for i in range(0,len(token_ids)-max_length,stride):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    ## to get particular row as input and output
    def  __getitem__(self,idx):
        return self.input_ids[idx],self.target_ids[idx]

In [8]:
### Batch Size Tells about the Number Of CPU's 
### num_workers:number of threads for each cpu
## drop_last : prevent the data if less than max_length

In [9]:
def create_dataloader_v1(txt, batch_size = 4,max_length = 256,
                        stride = 128,shuffle = True,drop_last=True,
                         num_workers = 0):

    #Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    #create a dataset.
    dataset = GPTDatasetV1(txt,tokenizer,max_length,stride)

    ## Create a Data Loader
    dataloader = DataLoader(dataset,batch_size=batch_size,
                            shuffle=shuffle,drop_last=drop_last,num_workers=num_workers)

    return dataloader
                        

In [10]:
## dataloader

dataloader = create_dataloader_v1(raw_text,batch_size=8,max_length=4,stride=4,shuffle=False)

### first batch input and output from the dataset
batch_iter = iter(dataloader)

first_batch = next(batch_iter)
print("First Batch:",first_batch)

First Batch: [tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]]), tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])]


In [12]:
inputs,targets = first_batch
print("Input:",inputs)
print("Size of Input:",inputs.shape)

Input: tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Size of Input: torch.Size([8, 4])


In [13]:
##each tensor contains 8 rows and 4 columns as input
#Lets create a token_embedding for each token id of dimensional of 256
#Each Input Batch becomes 8x4x256.

In [15]:
token_embedding = embedding_layer(inputs) 
### what we did here we create a token embedding,
#for each token from the input batch

print(token_embedding.shape)

torch.Size([8, 4, 256])


In [16]:
## Let's create a position Embedding layer, 
# for position it is common for each row from the input batch

## what size we required is 4x256 because we are using 256 dimensions and the size of each row
# from the inputs is 4 so 4x25

context_size = 4

pos_embedding_layer = nn.Embedding(context_size,out_dim)
print(pos_embedding_layer)

Embedding(4, 256)


In [20]:
#for each row in input , size is 4 --> let's take 0,1,2,3 these positions are fixed for every row in inputs

pos_embedding = pos_embedding_layer(torch.arange(4))
print(pos_embedding.shape)

torch.Size([4, 256])


In [ ]:
print(pos_embedding) # these are positional Embeddings for each position.

tensor([[-0.0472,  0.2852,  1.1994,  ...,  0.0850, -0.9262,  0.4838],
        [ 0.5746,  0.3778,  0.3025,  ...,  0.3413,  0.7450,  0.9606],
        [-0.4754,  0.5142, -2.0073,  ..., -1.7636,  0.8670,  1.4091],
        [-0.4613, -1.5100,  0.8002,  ..., -0.8700, -0.8618, -0.3774]],
       grad_fn=<EmbeddingBackward0>)


In [26]:
## add the token embedding + position Embedding
### What we are doing same position embeddings are added to each row from the token embedding

input_embeddings = token_embedding+pos_embedding
print(input_embeddings.shape)

torch.Size([8, 4, 256])


In [27]:
## These input embedding further training into the LLM